# T0101 — EDA dữ liệu UCI Beijing PM2.5

Notebook khám phá dữ liệu trước khi xử lý.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = Path('../data/raw/PRSA_data.csv')
OUT_DIR = Path('../outputs/eda')
OUT_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip().str.lower()
df.head()

In [ ]:
df.info()
display(df.describe(include='all').T)
print('Số dòng, số cột:', df.shape)
print('Số bản ghi trùng:', df.duplicated().sum())

## Missing values

In [ ]:
missing = df.isna().sum().to_frame('missing_count')
missing['missing_rate'] = df.isna().mean()
missing = missing.sort_values('missing_rate', ascending=False)
display(missing)
missing['missing_rate'].plot(kind='bar', figsize=(12, 4), title='Tỷ lệ Missing Values theo cột')
plt.tight_layout()
plt.savefig(OUT_DIR / 'missing_by_column.png', dpi=150)
plt.show()

In [ ]:
if {'year','month','day','hour'}.issubset(df.columns):
    df['datetime'] = pd.to_datetime(df[['year','month','day','hour']])
    monthly_missing = df.set_index('datetime').isna().resample('MS').mean()
    plt.figure(figsize=(14, 5))
    sns.heatmap(monthly_missing.T, cmap='viridis', cbar_kws={'label':'Missing rate'})
    plt.title('Missing Values theo tháng')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'missing_by_month.png', dpi=150)
    plt.show()

## Phân phối biến mục tiêu và class imbalance (nếu phân loại)

In [ ]:
TARGET = 'pm2.5'
if TARGET in df.columns:
    sns.histplot(df[TARGET].dropna(), bins=50, kde=True)
    plt.title('Phân phối PM2.5')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'target_distribution.png', dpi=150)
    plt.show()

# PM2.5 là bài toán hồi quy. Nếu nhóm chuyển thành phân loại, tạo nhãn theo ngưỡng phù hợp.
# Ví dụ minh họa phân lớp: thấp/vừa/cao; cần thống nhất ngưỡng với đề tài trước khi dùng.
if TARGET in df.columns:
    bins = [-float('inf'), 35, 75, float('inf')]
    labels = ['low', 'medium', 'high']
    classes = pd.cut(df[TARGET], bins=bins, labels=labels)
    print('Tỷ lệ lớp minh họa:')
    display(classes.value_counts(normalize=True).rename('ratio'))

## Chu kỳ ngày/đêm và độ dài chuỗi

In [ ]:
if 'hour' in df.columns and TARGET in df.columns:
    hourly = df.groupby('hour')[TARGET].mean()
    hourly.plot(figsize=(10, 4), marker='o')
    plt.xticks(range(0, 24))
    plt.title('PM2.5 trung bình theo giờ')
    plt.xlabel('Hour')
    plt.ylabel('PM2.5')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'day_night_cycle.png', dpi=150)
    plt.show()

if 'datetime' in df.columns:
    print('Khoảng thời gian:', df['datetime'].min(), '→', df['datetime'].max())
    print('Số bản ghi theo ngày:')
    display(df.set_index('datetime').resample('D').size().describe())